# Tarea 2 - Pregunta 2
## Construcción de la variable territorial y alistamiento de la base analítica

**Rol:** Ingeniería de datos
**Integrante:** Cristian Camilo Rodríguez Cagueñas

### Objetivo

En el notebook `01_auditoria_territorial_p2.ipynb` se estableció que el conjunto
de datos de contratación de INVIAS no contiene información estructurada sobre el
lugar de ejecución de los contratos, y que dicha información sí se encuentra
disponible de forma no estructurada en el objeto contractual.

Este notebook desarrolla la **construcción de la variable territorial** a partir
del texto libre, su validación, y el alistamiento de la base analítica que
sustenta la Pregunta de Negocio 2:

> ¿Dónde ejecuta INVIAS sus recursos? Dado que el SECOP II no registra el lugar
> de ejecución del contrato, ¿qué proporción de la contratación de la entidad es
> territorialmente identificable a partir del objeto contractual, y cómo se
> distribuyen y evolucionan en el territorio los recursos que sí lo son?

### Estructura del notebook

1. Carga y depuración de variables sin capacidad diferenciadora
2. Normalización del texto contractual
3. Construcción del diccionario territorial
4. Extracción del departamento
5. Extracción de la Dirección Territorial y del código de vía
6. Validación de la extracción
7. Tratamiento de contratos multidepartamentales
8. Construcción de indicadores y de la base analítica
9. Exportación

## 1. Carga y depuración de variables sin capacidad diferenciadora

In [1]:
# Importamos las librerías necesarias
import pandas as pd
import numpy as np
import re
import unicodedata

In [2]:
# Cargamos el conjunto de datos de INVIAS
df = pd.read_csv("../../invias.csv", low_memory=False)

print("Dimensiones iniciales:", df.shape)

Dimensiones iniciales: (25605, 89)


### 1.1 Eliminación de variables constantes

La auditoría estableció que `ciudad`, `localizacion` y `orden` presentan un único
valor en el 100 % de los registros, y que `direccion_de_ejecucion_del_contrato`
se encuentra vacía salvo en tres registros que además registran el valor
*"No definido"*.

La variable `departamento` también es constante. Sin embargo, **no se elimina**:
se conserva bajo el nombre `departamento_entidad` para dejar explícito en la base
analítica que la fuente sí registra un departamento, pero corresponde al
domicilio de la entidad y no al lugar de ejecución. Esta distinción es parte del
hallazgo y debe quedar trazable.

In [3]:
# Renombramos la variable departamento para explicitar su significado real
df = df.rename(columns={"departamento": "departamento_entidad"})

# Eliminamos las variables sin capacidad diferenciadora para el análisis
variables_a_eliminar = ["ciudad", "localizacion", "orden", "direccion_de_ejecucion_del_contrato"]
df = df.drop(columns=variables_a_eliminar)

print("Variables eliminadas:", variables_a_eliminar)
print("Dimensiones tras la depuración:", df.shape)

Variables eliminadas: ['ciudad', 'localizacion', 'orden', 'direccion_de_ejecucion_del_contrato']
Dimensiones tras la depuración: (25605, 85)


## 2. Normalización del texto contractual

La extracción por coincidencia textual requiere que el texto se encuentre en un
formato homogéneo. Se aplican las siguientes transformaciones:

- eliminación de tildes y caracteres diacríticos,
- conversión a mayúsculas,
- reemplazo de signos de puntuación por espacios,
- unificación de espacios múltiples.

Se conserva el texto original y la normalización se realiza sobre una variable
auxiliar, de manera que no se pierda información de la fuente.

In [4]:
def normalizar_texto(valor):
    """Normaliza el texto para permitir coincidencias exactas por diccionario.

    Elimina tildes, convierte a mayúsculas, reemplaza signos de puntuación por
    espacios y unifica espacios múltiples.
    """
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = texto.encode("ascii", "ignore").decode()
    texto = texto.upper()
    texto = re.sub(r"[^A-Z0-9 ]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()


# Concatenamos las dos variables que describen el alcance del contrato.
# Se usan ambas porque presentan patrones de completitud ligeramente distintos.
df["texto_contractual"] = (
    df["objeto_del_contrato"].fillna("") + " " + df["descripcion_del_proceso"].fillna("")
).map(normalizar_texto)

print("Registros con texto disponible:", (df["texto_contractual"].str.len() > 0).sum())
print()
print("Ejemplo de normalización:")
print(df["texto_contractual"].dropna().iloc[1][:200])

Registros con texto disponible: 20717

Ejemplo de normalización:
PRESTAR SERVICIOS PROFESIONALES PARA ATENDER ASUNTOS DE CARACTER SOCIAL DEL PROGRAMA CAMINOS COMUNITARIOS DE LA PAZ TOTAL EN LA DIRECCION TERRITORIAL CAUCA PRESTAR SERVICIOS PROFESIONALES PARA ATENDER


## 3. Construcción del diccionario territorial

### 3.1 Diccionario de departamentos y alias

Se construye un diccionario que asocia cada uno de los 32 departamentos de
Colombia y Bogotá D.C. con las formas textuales bajo las cuales pueden aparecer
en el objeto contractual.

Dos decisiones metodológicas relevantes:

**Precedencia por longitud.** El alias `NORTE DE SANTANDER` contiene la cadena
`SANTANDER`. Si se evaluaran de forma independiente, un contrato de Norte de
Santander sería asignado también a Santander. Para evitarlo, los alias se
ordenan de mayor a menor longitud y, una vez detectado un alias, su ocurrencia se
**consume** del texto antes de evaluar los alias restantes. El mismo criterio
aplica a `SAN ANDRES` y `LA GUAJIRA`.

**Exclusión de alias ambiguos.** No se incluye el alias `VALLE` de forma aislada,
por su alta frecuencia en expresiones geográficas comunes no relacionadas con el
departamento (por ejemplo, *"valle del río"*). Únicamente se admite
`VALLE DEL CAUCA`.

In [5]:
# Diccionario de departamentos y sus formas textuales admitidas
DEPARTAMENTOS = {
    "Amazonas": ["AMAZONAS"],
    "Antioquia": ["ANTIOQUIA"],
    "Arauca": ["ARAUCA"],
    "Atlántico": ["ATLANTICO"],
    "Bolívar": ["BOLIVAR"],
    "Boyacá": ["BOYACA"],
    "Caldas": ["CALDAS"],
    "Caquetá": ["CAQUETA"],
    "Casanare": ["CASANARE"],
    "Cauca": ["CAUCA"],
    "Cesar": ["CESAR"],
    "Chocó": ["CHOCO"],
    "Córdoba": ["CORDOBA"],
    "Cundinamarca": ["CUNDINAMARCA"],
    "Guainía": ["GUAINIA"],
    "Guaviare": ["GUAVIARE"],
    "Huila": ["HUILA"],
    "La Guajira": ["LA GUAJIRA", "GUAJIRA"],
    "Magdalena": ["MAGDALENA"],
    "Meta": ["META"],
    "Nariño": ["NARINO"],
    "Norte de Santander": ["NORTE DE SANTANDER", "N DE SANTANDER"],
    "Putumayo": ["PUTUMAYO"],
    "Quindío": ["QUINDIO"],
    "Risaralda": ["RISARALDA"],
    "San Andrés y Providencia": [
        "SAN ANDRES Y PROVIDENCIA",
        "ARCHIPIELAGO DE SAN ANDRES",
        "SAN ANDRES ISLAS",
        "PROVIDENCIA Y SANTA CATALINA",
    ],
    "Santander": ["SANTANDER"],
    "Sucre": ["SUCRE"],
    "Tolima": ["TOLIMA"],
    "Valle del Cauca": ["VALLE DEL CAUCA"],
    "Vaupés": ["VAUPES"],
    "Vichada": ["VICHADA"],
    "Bogotá D.C.": ["BOGOTA"],
}

# Ordenamos los alias de mayor a menor longitud para dar precedencia
# a las formas más específicas (p. ej. NORTE DE SANTANDER antes que SANTANDER)
ALIAS_ORDENADOS = sorted(
    [(alias, dep) for dep, alias_list in DEPARTAMENTOS.items() for alias in alias_list],
    key=lambda x: -len(x[0]),
)

PATRONES = [
    (re.compile(r"\b" + re.escape(alias) + r"\b"), dep)
    for alias, dep in ALIAS_ORDENADOS
]

print("Departamentos en el diccionario:", len(DEPARTAMENTOS))
print("Alias totales:", len(ALIAS_ORDENADOS))


def extraer_departamentos(texto):
    """Devuelve la lista de departamentos mencionados en el texto.

    Al detectar un alias, su ocurrencia se elimina del texto restante para evitar
    que alias contenidos en otros (p. ej. SANTANDER dentro de NORTE DE SANTANDER)
    generen asignaciones duplicadas.
    """
    encontrados = []
    restante = texto
    for patron, departamento in PATRONES:
        if patron.search(restante):
            encontrados.append(departamento)
            restante = patron.sub(" ", restante)
    return sorted(set(encontrados))


Departamentos en el diccionario: 33
Alias totales: 38


### 3.2 Tratamiento de topónimos ambiguos

La revisión manual de la muestra de validación (sección 6.2) permitió identificar
un mecanismo sistemático de **falsos positivos**: existen nombres de municipios y
de regiones que contienen el nombre de un departamento al que no pertenecen.

Los casos identificados y su generalización:

| Expresión | Departamento asignado erróneamente | Ubicación real |
|---|---|---|
| `SAN JUAN DEL CESAR` | Cesar | Municipio de **La Guajira** |
| `MAGDALENA MEDIO` | Magdalena | **Región** que abarca Santander, Antioquia, Bolívar y Cesar |
| `CIUDAD BOLIVAR` | Bolívar | Localidad de Bogotá / municipio de Antioquia |
| `MUNICIPIO DE BOLIVAR` | Bolívar | Municipio homónimo en Cauca, Valle o Santander |
| `MUNICIPIO DE SUCRE` | Sucre | Municipio homónimo en Santander o Cauca |
| `MUNICIPIO DE CORDOBA` | Córdoba | Municipio homónimo en Bolívar, Nariño o Quindío |

El tratamiento consiste en **enmascarar estas expresiones antes de aplicar el
diccionario**, y se mide su efecto comparando la asignación con y sin enmascaramiento. El enmascaramiento es local a la expresión: si el departamento se
menciona además en otra parte del texto, la asignación se conserva. Esto evita
descartar contratos que sí corresponden al departamento.

In [6]:
# Expresiones que contienen el nombre de un departamento al que no pertenecen.
# Se enmascaran antes de aplicar el diccionario para evitar falsos positivos.
TOPONIMOS_AMBIGUOS = {
    "SAN JUAN DEL CESAR":   "municipio de La Guajira",
    "MAGDALENA MEDIO":      "región de cuatro departamentos",
    "CIUDAD BOLIVAR":       "localidad de Bogotá",
    "MUNICIPIO DE BOLIVAR": "homónimo en Cauca, Valle o Santander",
    "MUNICIPIO DE SUCRE":   "homónimo en Santander o Cauca",
    "MUNICIPIO DE CORDOBA": "homónimo en Bolívar, Nariño o Quindío",
}

# Grupo sin captura: evita la advertencia de pandas al usarlo en str.contains
PATRON_AMBIGUOS = re.compile(
    r"\b(?:" + "|".join(re.escape(t) for t in TOPONIMOS_AMBIGUOS) + r")\b"
)


def enmascarar_ambiguos(texto):
    """Reemplaza los topónimos ambiguos por un marcador neutro."""
    return PATRON_AMBIGUOS.sub(" TOPONIMO_AMBIGUO ", texto)


# Aplicamos el enmascaramiento y medimos su efecto sobre la asignación
df["texto_para_extraccion"] = df["texto_contractual"].map(enmascarar_ambiguos)

sin_tratamiento = df["texto_contractual"].map(extraer_departamentos)
con_tratamiento = df["texto_para_extraccion"].map(extraer_departamentos)
corregidas = (sin_tratamiento != con_tratamiento).sum()

print("Tratamiento de topónimos ambiguos")
print()
print("Contratos que contienen algún topónimo ambiguo:",
      df["texto_contractual"].str.contains(PATRON_AMBIGUOS, regex=True).sum())
print()
for toponimo, ubicacion_real in TOPONIMOS_AMBIGUOS.items():
    conteo = df["texto_contractual"].str.contains(r"\b" + toponimo + r"\b", regex=True).sum()
    print(f"  {toponimo:<22} {conteo:>4}   ({ubicacion_real})")
print()
print("Asignaciones corregidas por el tratamiento:", corregidas)

Tratamiento de topónimos ambiguos

Contratos que contienen algún topónimo ambiguo: 98



  SAN JUAN DEL CESAR       44   (municipio de La Guajira)
  MAGDALENA MEDIO          17   (región de cuatro departamentos)


  CIUDAD BOLIVAR           10   (localidad de Bogotá)
  MUNICIPIO DE BOLIVAR     15   (homónimo en Cauca, Valle o Santander)
  MUNICIPIO DE SUCRE        9   (homónimo en Santander o Cauca)


  MUNICIPIO DE CORDOBA      3   (homónimo en Bolívar, Nariño o Quindío)

Asignaciones corregidas por el tratamiento: 80


## 4. Extracción del departamento

In [7]:
df["departamentos_detectados"] = df["texto_para_extraccion"].map(extraer_departamentos)
df["n_departamentos"] = df["departamentos_detectados"].map(len)

df["n_departamentos"].value_counts().sort_index().head(8)

n_departamentos
0    16927
1     8214
2      411
3       37
4       13
6        1
7        2
Name: count, dtype: int64

### 4.1 Clasificación de la trazabilidad territorial

Se construye la variable `trazabilidad_territorial`, que clasifica cada contrato
en tres categorías según la información territorial recuperada.

In [8]:
# Clasificamos los contratos según la información territorial recuperada
df["trazabilidad_territorial"] = np.select(
    [df["n_departamentos"] == 0, df["n_departamentos"] == 1],
    ["No territorializable", "Departamento único"],
    default="Multidepartamental",
)

# Variable de departamento inferido: solo se asigna cuando la atribución es inequívoca
df["departamento_inferido"] = np.where(
    df["n_departamentos"] == 1,
    df["departamentos_detectados"].str[0],
    np.nan,
)

resumen = df["trazabilidad_territorial"].value_counts().to_frame("contratos")
resumen["%"] = (100 * resumen["contratos"] / len(df)).round(1)
resumen

,contratos,%
trazabilidad_territorial,,
No territorializable,16927,66.1
Departamento único,8214,32.1
Multidepartamental,464,1.8


## 5. Extracción de la Dirección Territorial y del código de vía

Además del departamento, el objeto contractual contiene dos elementos adicionales
de valor territorial:

- **Dirección Territorial**: unidad administrativa del INVIAS responsable de la
  gestión de la vía. Corresponde a un concepto **distinto** al departamento de
  ejecución: una Dirección Territorial puede administrar corredores que atraviesan
  varios departamentos.
- **Código de vía, ruta nacional y abscisado (PR)**: identifican el corredor vial
  específico y el tramo intervenido. La entidad los referencia bajo formatos
  distintos, por lo que se contemplan las tres variantes.

Ambos se extraen como variables independientes.

In [9]:
# Extracción de la Dirección Territorial
PATRON_DT = re.compile(r"DIRECCION TERRITORIAL\s+([A-Z]+(?:\s+DE\s+[A-Z]+)?(?:\s+[A-Z]+)?)")


def extraer_direccion_territorial(texto):
    """Identifica la Dirección Territorial del INVIAS mencionada en el texto."""
    coincidencia = PATRON_DT.search(texto)
    if not coincidencia:
        return np.nan
    candidato = coincidencia.group(1)
    for patron, departamento in PATRONES:
        if patron.search(candidato):
            return departamento
    return np.nan


df["direccion_territorial"] = df["texto_contractual"].map(extraer_direccion_territorial)

print("Contratos con Dirección Territorial identificada:",
      df["direccion_territorial"].notna().sum())
print()
print(df["direccion_territorial"].value_counts().head(10).to_string())

Contratos con Dirección Territorial identificada: 3306

direccion_territorial
Boyacá                225
Santander             202
Nariño                192
Cauca                 187
Meta                  179
Antioquia             167
Norte de Santander    166
Cundinamarca          161
Huila                 158
Córdoba               155


In [10]:
# Extracción del código de vía, la ruta nacional y el abscisado (PR)
# El objeto contractual referencia el corredor de varias formas: "Vía Código: 45A08",
# "código 47104" y "Ruta 45". Se contemplan las tres.
PATRON_CODIGO = re.compile(r"CODIGO\s+([0-9]{2,5}[A-Z]{0,3}[0-9]{0,3})")
PATRON_RUTA = re.compile(r"\bRUTA\s+([0-9]{2,4}[A-Z]{0,2})")
PATRON_PR = re.compile(r"\bPR\s?(?:[0-9]{1,4})")

df["codigo_via"] = df["texto_contractual"].str.extract(PATRON_CODIGO)[0]
df["ruta_nacional"] = df["texto_contractual"].str.extract(PATRON_RUTA)[0]
df["tiene_abscisado"] = df["texto_contractual"].str.contains(PATRON_PR, regex=True)

print("Contratos con código de vía identificado:", df["codigo_via"].notna().sum())
print("Contratos con ruta nacional identificada:", df["ruta_nacional"].notna().sum())
print("Contratos con abscisado (PR) identificado:", df["tiene_abscisado"].sum())
print()
print("Contratos con al menos una referencia al corredor vial:",
      (df["codigo_via"].notna() | df["ruta_nacional"].notna() | df["tiene_abscisado"]).sum())

Contratos con código de vía identificado: 413
Contratos con ruta nacional identificada: 641
Contratos con abscisado (PR) identificado: 1912

Contratos con al menos una referencia al corredor vial: 2340


## 6. Validación de la extracción

### 6.1 Sobre la no validez de la Dirección Territorial como criterio de validación

Una alternativa aparentemente razonable sería validar el departamento extraído
contra la Dirección Territorial, verificando si ambos coinciden.

**Este procedimiento no es válido en este caso**, por dos razones:

1. **Circularidad.** La frase *"Dirección Territorial X"* forma parte del mismo
   texto sobre el cual se aplica el diccionario. El departamento de la Dirección
   Territorial es detectado por el extractor precisamente porque aparece en esa
   frase, de modo que la coincidencia es de casi el 100 % por construcción y no
   mide precisión alguna.

2. **Diferencia conceptual.** La Dirección Territorial identifica la unidad
   administrativa que gestiona la vía, no el departamento donde esta se ubica.
   Una misma Dirección Territorial administra corredores que atraviesan varios
   departamentos.

Se demuestra a continuación el primer punto de forma explícita, eliminando la
frase de la Dirección Territorial del texto antes de reaplicar el extractor.

In [11]:
# Demostración de la circularidad y de la diferencia conceptual
PATRON_DT_FRASE = re.compile(r"DIRECCION TERRITORIAL\s+[A-Z]+(?:\s+DE\s+[A-Z]+)?(?:\s+[A-Z]+)?")

con_dt = df[df["direccion_territorial"].notna()].copy()

# (a) Comparación directa: circular
coincide_directo = con_dt.apply(
    lambda fila: fila["direccion_territorial"] in fila["departamentos_detectados"], axis=1
)
print("Comparación directa (circular):")
print(f"  coincidencia: {100 * coincide_directo.mean():.1f} %  sobre {len(con_dt)} contratos")
print()

# (b) Comparación tras eliminar la frase de la Dirección Territorial del texto
texto_sin_dt = con_dt["texto_contractual"].str.replace(PATRON_DT_FRASE, " ", regex=True)
deps_sin_dt = texto_sin_dt.map(extraer_departamentos)

menciona_otro = deps_sin_dt.map(len) > 0
print("Comparación independiente (frase de la Dirección Territorial eliminada):")
print(f"  contratos cuyo texto restante menciona algún departamento: {menciona_otro.sum()}"
      f" ({100 * menciona_otro.mean():.1f} %)")

coincide_indep = pd.Series(
    [dt in deps for dt, deps in zip(con_dt["direccion_territorial"][menciona_otro],
                                     deps_sin_dt[menciona_otro])]
)
print(f"  de esos, coinciden con la Dirección Territorial: {coincide_indep.sum()}"
      f" ({100 * coincide_indep.mean():.1f} %)")

Comparación directa (circular):
  coincidencia: 100.0 %  sobre 3306 contratos



Comparación independiente (frase de la Dirección Territorial eliminada):
  contratos cuyo texto restante menciona algún departamento: 292 (8.8 %)
  de esos, coinciden con la Dirección Territorial: 186 (63.7 %)


El resultado confirma ambos puntos: la comparación directa arroja una
coincidencia prácticamente total, mientras que la comparación independiente
arroja un nivel de acuerdo sustancialmente menor sobre una muestra reducida.

Los desacuerdos observados no corresponden mayoritariamente a errores del
extractor, sino a la diferencia conceptual señalada: contratos gestionados por
una Dirección Territorial cuya intervención se localiza en un departamento
vecino. Por tanto, **ambas variables se conservan como información complementaria
y no se utiliza una para validar la otra**.

### 6.2 Validación por muestreo aleatorio

La precisión de la extracción se estima mediante revisión manual de una muestra
aleatoria de contratos con departamento único inferido. Para cada contrato de la
muestra se registra el objeto contractual y el departamento asignado, y se
verifica manualmente si la asignación es correcta.

La muestra se genera con semilla fija para garantizar la reproducibilidad y se
exporta a `muestra_validacion_p2.csv` para su revisión.

In [12]:
# Generamos una muestra aleatoria reproducible para validación manual
TAMANO_MUESTRA = 60
SEMILLA = 42

muestra_validacion = (
    df[df["trazabilidad_territorial"] == "Departamento único"]
    .sample(TAMANO_MUESTRA, random_state=SEMILLA)[
        ["id_contrato", "objeto_del_contrato", "departamento_inferido"]
    ]
    .reset_index(drop=True)
)

# Columnas para el diligenciamiento manual
muestra_validacion["asignacion_correcta"] = ""   # SI / NO
muestra_validacion["observacion"] = ""

muestra_validacion.to_csv("muestra_validacion_p2.csv", index=False, encoding="utf-8-sig")

print(f"Muestra de {TAMANO_MUESTRA} contratos exportada a muestra_validacion_p2.csv")
muestra_validacion.head(10)

Muestra de 60 contratos exportada a muestra_validacion_p2.csv


,id_contrato,objeto_del_contrato,departamento_inferido,asignacion_correcta,observacion
0,CO1.PCCNTR.7129274,INTERVENTORIA PARA LA ATENCIÓN DE LA EMERGENCI...,Norte de Santander,,
1,CO1.PCCNTR.6856890,SUMINISTRO MEZCLA ASFÁLTICA RUTA 9004 VÍA LOR...,Sucre,,
2,CO1.PCCNTR.3709003,HABILITAR Y/O BRINDAR TRANSITABILIDAD EN LA RE...,Santander,,
3,CO1.PCCNTR.5354492,AUNAR ESFUERZOS ENTRE EL INSTITUTO NACIONAL DE...,Nariño,,
4,CO1.PCCNTR.1038819,INTERVENTORIA A LA CONSTRUCCION DEL MUELLE DE ...,Chocó,,
5,CO1.PCCNTR.5537189,AUNAR ESFUERZOS ENTRE EL INSTITUTO NACIONAL DE...,Antioquia,,
6,CO1.PCCNTR.5417816,AUNAR ESFUERZOS ENTRE EL INSTITUTO NACIONAL DE...,Tolima,,
7,CO1.PCCNTR.1481706,INTERVENTORIA TECNICA; ADMINISTRATIVA; FINANCI...,Nariño,,
8,CO1.PCCNTR.5310641,SERVICIO DE MANTENIMIENTO RUTINARIO DE LAS VIA...,Huila,,
9,CO1.PCCNTR.7753254,AUNAR ESFUERZOS TÉCNICOS; ADMINISTRATIVOS; JUR...,Boyacá,,


### 6.3 Verificación de consistencia interna

De forma complementaria a la revisión manual, se verifica que la extracción no
produzca asignaciones internamente inconsistentes.

In [13]:
# Verificamos que ningún contrato tenga departamento inferido sin haber sido clasificado como único
inconsistentes = df[
    (df["departamento_inferido"].notna()) & (df["trazabilidad_territorial"] != "Departamento único")
]
print("Contratos con departamento inferido e inconsistencia de clasificación:", len(inconsistentes))

# Verificamos que ningún contrato sin texto haya recibido asignación territorial
sin_texto_con_dep = df[(df["texto_contractual"].str.len() == 0) & (df["n_departamentos"] > 0)]
print("Contratos sin texto con asignación territorial:", len(sin_texto_con_dep))

# Verificamos que los departamentos asignados pertenezcan al diccionario
asignados = set(df["departamento_inferido"].dropna().unique())
print("Departamentos asignados fuera del diccionario:", asignados - set(DEPARTAMENTOS.keys()))
print("Departamentos distintos identificados:", len(asignados))

Contratos con departamento inferido e inconsistencia de clasificación: 0
Contratos sin texto con asignación territorial: 0
Departamentos asignados fuera del diccionario: set()
Departamentos distintos identificados: 33


## 7. Tratamiento de contratos multidepartamentales

Los contratos que mencionan más de un departamento presentan una decisión
metodológica relevante. Existen dos alternativas:

1. **Replicar** el contrato en cada departamento mencionado. Esta opción permite
   que cada departamento aparezca en los conteos, pero **duplica el valor
   contratado** en las agregaciones, inflando el total nacional.

2. **Tratarlos como categoría propia**, sin asignarlos a ningún departamento
   individual.

Se adopta la **segunda alternativa**, dado que la pregunta de negocio se refiere a
la distribución de *recursos* y la duplicación de valor invalidaría cualquier
ranking o participación porcentual. Los contratos multidepartamentales se
reportan de manera explícita como categoría separada, preservando su peso en el
total.

Se cuantifica a continuación la magnitud de esta decisión.

In [14]:
# Cuantificamos el peso de los contratos multidepartamentales
UMBRAL_VALOR_ATIPICO = 1e12

df_valor = df[
    df["valor_del_contrato"].notna()
    & (df["valor_del_contrato"] > 0)
    & (df["valor_del_contrato"] < UMBRAL_VALOR_ATIPICO)
].copy()

peso = df_valor.groupby("trazabilidad_territorial").agg(
    contratos=("id_contrato", "size"),
    valor_total=("valor_del_contrato", "sum"),
)
peso["%_contratos"] = (100 * peso["contratos"] / peso["contratos"].sum()).round(1)
peso["valor_billones"] = (peso["valor_total"] / 1e12).round(2)
peso["%_valor"] = (100 * peso["valor_total"] / peso["valor_total"].sum()).round(1)

peso[["contratos", "%_contratos", "valor_billones", "%_valor"]]

,contratos,%_contratos,valor_billones,%_valor
trazabilidad_territorial,,,,
Departamento único,8029,40.9,19.99,51.1
Multidepartamental,457,2.3,10.50,26.8
No territorializable,11131,56.7,8.64,22.1


In [15]:
# Revisamos cuántos departamentos suelen mencionar los contratos multidepartamentales
multi = df[df["trazabilidad_territorial"] == "Multidepartamental"]
print("Distribución del número de departamentos mencionados:")
print(multi["n_departamentos"].value_counts().sort_index().to_string())

Distribución del número de departamentos mencionados:
n_departamentos
2    411
3     37
4     13
6      1
7      2


## 8. Construcción de indicadores y de la base analítica

### 8.1 Variables temporales

Se convierten las fechas contractuales al tipo `datetime` y se derivan las
variables de año y trimestre de firma, necesarias para el análisis de evolución
temporal. Se replica el criterio aplicado en la Pregunta 3: se conservan los
valores faltantes sin imputación.

In [16]:
# Convertimos las variables de fecha al formato datetime
columnas_fecha = ["fecha_de_firma", "fecha_de_inicio_del_contrato", "fecha_de_fin_del_contrato"]

for columna in columnas_fecha:
    no_nulos_antes = df[columna].notna().sum()
    df[columna] = pd.to_datetime(df[columna], errors="coerce", utc=True)
    no_nulos_despues = df[columna].notna().sum()
    print(f"{columna}: {no_nulos_antes} no nulos antes -> {no_nulos_despues} después")

# Derivamos variables temporales a partir de la fecha de firma
df["anio_firma"] = df["fecha_de_firma"].dt.year
df["trimestre_firma"] = df["fecha_de_firma"].dt.tz_localize(None).dt.to_period("Q").astype(str)

fecha_de_firma: 17477 no nulos antes -> 17477 después
fecha_de_inicio_del_contrato: 17795 no nulos antes -> 17795 después
fecha_de_fin_del_contrato: 20074 no nulos antes -> 20074 después


### 8.2 Indicador de trazabilidad territorial

Se construye el indicador central de la pregunta reformulada: la proporción de
recursos que resulta territorialmente identificable, calculada por año de firma.

In [17]:
# Indicador binario de territorializable
df["es_territorializable"] = df["trazabilidad_territorial"] != "No territorializable"

# Evolución anual de la trazabilidad territorial (sobre contratos con valor válido)
df_valor = df[
    df["valor_del_contrato"].notna()
    & (df["valor_del_contrato"] > 0)
    & (df["valor_del_contrato"] < UMBRAL_VALOR_ATIPICO)
].copy()

trazabilidad_anual = df_valor.groupby("anio_firma").apply(
    lambda g: pd.Series({
        "contratos": len(g),
        "%_contratos_territorializables": round(100 * g["es_territorializable"].mean(), 1),
        "valor_billones": round(g["valor_del_contrato"].sum() / 1e12, 2),
        "%_valor_territorializable": round(
            100 * g.loc[g["es_territorializable"], "valor_del_contrato"].sum()
            / g["valor_del_contrato"].sum(), 1),
    }),
    include_groups=False,
)

trazabilidad_anual

,contratos,%_contratos_territorializables,valor_billones,%_valor_territorializable
anio_firma,,,,
2017.0,47.0,46.8,0.00,28.8
2018.0,1070.0,37.6,1.66,94.1
2019.0,1253.0,55.1,1.65,73.4
2020.0,2037.0,46.8,4.12,85.3
2021.0,2019.0,41.3,12.87,94.7
2022.0,1504.0,38.3,2.13,93.1
2023.0,3871.0,50.7,1.56,76.0
2024.0,2952.0,58.0,2.56,76.3
2025.0,1613.0,27.6,0.92,46.3


### 8.3 Distribución territorial de los recursos

Se construye la agregación por departamento sobre los contratos con atribución
territorial inequívoca.

In [18]:
# Agregación por departamento (solo contratos con departamento único)
por_departamento = (
    df_valor[df_valor["trazabilidad_territorial"] == "Departamento único"]
    .groupby("departamento_inferido")
    .agg(
        contratos=("id_contrato", "size"),
        valor_total=("valor_del_contrato", "sum"),
        valor_mediano=("valor_del_contrato", "median"),
    )
)
por_departamento["valor_miles_millones"] = (por_departamento["valor_total"] / 1e9).round(1)
por_departamento["%_valor"] = (
    100 * por_departamento["valor_total"] / por_departamento["valor_total"].sum()
).round(1)

por_departamento = por_departamento.sort_values("valor_total", ascending=False)
por_departamento[["contratos", "valor_miles_millones", "%_valor"]].head(15)

,contratos,valor_miles_millones,%_valor
departamento_inferido,,,
Antioquia,475,2564.5,12.8
Nariño,482,2148.9,10.7
Cauca,465,2025.8,10.1
Putumayo,210,1476.3,7.4
Norte de Santander,387,1295.4,6.5
Boyacá,521,1152.9,5.8
Córdoba,427,978.9,4.9
Santander,418,973.3,4.9
Tolima,334,818.9,4.1


### 8.4 Selección de variables de la base analítica

Se conservan las variables necesarias para responder la Pregunta 2: las
territoriales construidas, las contractuales, las temporales y las financieras
básicas requeridas para dimensionar los recursos.

In [19]:
# Definimos las variables que conforman la base analítica de la Pregunta 2
variables_base_p2 = [
    # Identificación
    "id_contrato", "referencia_del_contrato",
    # Territoriales (construidas)
    "trazabilidad_territorial", "departamento_inferido", "n_departamentos",
    "direccion_territorial", "codigo_via", "ruta_nacional", "tiene_abscisado",
    "es_territorializable",
    # Territorial de la fuente (domicilio de la entidad)
    "departamento_entidad",
    # Contractuales
    "estado_contrato", "tipo_de_contrato", "modalidad_de_contratacion",
    "proveedor_adjudicado", "destino_gasto", "origen_de_los_recursos",
    # Temporales
    "fecha_de_firma", "anio_firma", "trimestre_firma",
    "fecha_de_inicio_del_contrato", "fecha_de_fin_del_contrato", "duracion_del_contrato",
    # Financieras
    "valor_del_contrato", "valor_pagado", "valor_pendiente_de_ejecucion",
    # Texto de respaldo para trazabilidad de la inferencia
    "objeto_del_contrato",
]

df_analitico_p2 = df[variables_base_p2].copy()

print("Dimensiones de la base analítica de la Pregunta 2:", df_analitico_p2.shape)
df_analitico_p2.head()

Dimensiones de la base analítica de la Pregunta 2: (25605, 27)


,id_contrato,referencia_del_contrato,trazabilidad_territorial,departamento_inferido,n_departamentos,direccion_territorial,codigo_via,ruta_nacional,tiene_abscisado,es_territorializable,...,fecha_de_firma,anio_firma,trimestre_firma,fecha_de_inicio_del_contrato,fecha_de_fin_del_contrato,duracion_del_contrato,valor_del_contrato,valor_pagado,valor_pendiente_de_ejecucion,objeto_del_contrato
0,CO1.PCCNTR.770322,000168-2019,No territorializable,NaN,0,NaN,NaN,NaN,False,False,...,2019-01-29 00:00:00+00:00,2019.0,2019Q1,2019-01-18 00:00:00+00:00,2019-07-17 00:00:00+00:00,Dia(s),23500000.0,22420440.0,1079559.0,PRESTAR LOS SERVICIOS PROFESIONALES EN TEMAS T...
1,CO1.PCCNTR.5481469,4028 DE 2023,Departamento único,Cauca,1,Cauca,NaN,NaN,False,True,...,2023-10-23 00:00:00+00:00,2023.0,2023Q4,2023-10-31 00:00:00+00:00,2023-12-31 00:00:00+00:00,67 Dia(s),15770000.0,0.0,15770000.0,PRESTAR SERVICIOS PROFESIONALES PARA ATENDER A...
2,CO1.PCCNTR.6706615,3429 DE 2024,No territorializable,NaN,0,NaN,NaN,NaN,False,False,...,2024-08-30 00:00:00+00:00,2024.0,2024Q3,2024-09-06 00:00:00+00:00,2024-12-31 00:00:00+00:00,120 Dia(s),31500000.0,19833333.0,11666667.0,PRESTACION DE SERVICIOS PROFESIONALES PARA BRI...
3,CO1.PCCNTR.4412417,265 DE 2023,No territorializable,NaN,0,NaN,NaN,NaN,False,False,...,2023-01-16 00:00:00+00:00,2023.0,2023Q1,2023-01-19 00:00:00+00:00,2023-04-19 00:00:00+00:00,3 Mes(es),17700000.0,0.0,17700000.0,PRESTAR SERVICIOS PROFESIONALES; PARA ATENDER ...
4,CO1.PCCNTR.3516681,CO1.PCCNTR.3516681,No territorializable,NaN,0,NaN,NaN,NaN,False,False,...,NaT,NaN,NaN,NaT,2022-09-27 00:00:00+00:00,8 Mes(es),0.0,0.0,0.0,No definido


## 9. Exportación y cierre del alistamiento

In [20]:
# Exportamos la base analítica y las agregaciones de apoyo
df_analitico_p2.to_csv("df_analitico_p2.csv", index=False, encoding="utf-8-sig")
por_departamento.to_csv("agregado_departamento_p2.csv", encoding="utf-8-sig")
trazabilidad_anual.to_csv("trazabilidad_anual_p2.csv", encoding="utf-8-sig")

print("Archivos generados:")
print("  df_analitico_p2.csv        ->", df_analitico_p2.shape)
print("  agregado_departamento_p2.csv ->", por_departamento.shape)
print("  trazabilidad_anual_p2.csv  ->", trazabilidad_anual.shape)
print("  muestra_validacion_p2.csv  -> muestra para validación manual")

Archivos generados:
  df_analitico_p2.csv        -> (25605, 27)
  agregado_departamento_p2.csv -> (33, 5)
  trazabilidad_anual_p2.csv  -> (10, 4)
  muestra_validacion_p2.csv  -> muestra para validación manual


### Cierre del alistamiento

La etapa de selección, limpieza y alistamiento de la Pregunta 2 permitió
establecer lo siguiente:

1. **Depuración.** Se eliminaron las variables `ciudad`, `localizacion`, `orden` y
   `direccion_de_ejecucion_del_contrato` por carecer de capacidad diferenciadora.
   La variable `departamento` se conservó bajo el nombre `departamento_entidad`
   para dejar trazable que corresponde al domicilio de la entidad.

2. **Construcción de la variable territorial.** Se implementó un extractor basado en
   un diccionario de 33 entidades territoriales con manejo de precedencia por
   longitud de alias, aplicado sobre el objeto contractual y la descripción del
   proceso normalizados.

3. **Variables complementarias.** Se extrajeron la Dirección Territorial del INVIAS y
   el código de vía como información territorial adicional, conservándolas como
   variables independientes por corresponder a conceptos distintos del departamento
   de ejecución.

4. **Validación.** Se descartó la validación cruzada contra la Dirección Territorial
   por circularidad y por diferencia conceptual, y se estableció en su lugar un
   procedimiento de validación por muestreo aleatorio reproducible.

5. **Tratamiento de multidepartamentales.** Se optó por no replicar los contratos
   entre departamentos, con el fin de no duplicar valor en las agregaciones, y se
   reportan como categoría propia.

6. **Base analítica.** Se generó `df_analitico_p2.csv`, que incorpora las variables
   territoriales construidas junto con las contractuales, temporales y financieras
   necesarias para el análisis.

**Siguiente paso:** exploración y análisis de datos de la Pregunta 2, desarrollada
en `Tarea 3/Pregunta 2/01_exploracion_p2.ipynb`.

**Limitación pendiente:** la revisión manual de `muestra_validacion_p2.csv` debe
completarse para reportar la precisión estimada del extractor en el informe final.